# Эксперименты и сравнение моделей

In [1]:
import sys
from pathlib import Path

import pandas as pd
from joblib import dump
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

PROJECT_ROOT = Path('..').resolve()
sys.path.append(str(PROJECT_ROOT))

from src.modeling import build_model_pipeline, evaluate_classifier, load_raw_data, split_data
from src.preprocessing import TARGET_COLUMN, prepare_data

RANDOM_STATE = 42
MODEL_PATH = PROJECT_ROOT / 'models' / 'best_model_cp1.joblib'
METRICS_PATH = PROJECT_ROOT / 'data' / 'processed' / 'metrics_cp1.csv'

Сначала повторяем одну и ту же подготовку данных, чтобы все модели сравнивались в одинаковых условиях.

In [2]:
prepared_df = prepare_data(load_raw_data())
train_df, val_df, test_df = split_data(prepared_df)

x_train = train_df.drop(columns=[TARGET_COLUMN])
y_train = train_df[TARGET_COLUMN]
x_val = val_df.drop(columns=[TARGET_COLUMN])
y_val = val_df[TARGET_COLUMN]

train_df.shape, val_df.shape, test_df.shape

((60466, 38), (12957, 38), (12957, 38))

Чтобы сравнение было честным, добавим в таблицу и baseline-модель `LogisticRegression`. Это позволяет увидеть улучшения, относительно базовой модели.

In [3]:
logistic_model = build_model_pipeline(
    train_df,
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
)
logistic_model.fit(x_train, y_train)
logistic_metrics = evaluate_classifier(logistic_model, x_val, y_val)
logistic_metrics

{'accuracy': 0.7968665586169638,
 'precision': 0.6841477949940405,
 'recall': 0.48384377634166903,
 'f1': 0.5668202764976958,
 'roc_auc': 0.8520672198881817}

## KNeighborsClassifier

In [4]:
knn_model = build_model_pipeline(
    train_df,
    KNeighborsClassifier(n_neighbors=15),
)
knn_model.fit(x_train, y_train)
knn_metrics = evaluate_classifier(knn_model, x_val, y_val)
knn_metrics

{'accuracy': 0.7990275526742301,
 'precision': 0.6696269982238011,
 'recall': 0.529643158190503,
 'f1': 0.5914653278945717,
 'roc_auc': 0.8391719292950065}

## Decision Tree

In [5]:
tree_model = build_model_pipeline(
    train_df,
    DecisionTreeClassifier(max_depth=8, random_state=RANDOM_STATE),
)
tree_model.fit(x_train, y_train)
tree_metrics = evaluate_classifier(tree_model, x_val, y_val)
tree_metrics

{'accuracy': 0.8153893648221039,
 'precision': 0.6923837784371909,
 'recall': 0.5900533857825232,
 'f1': 0.6371359223300971,
 'roc_auc': 0.8709592249724509}

## Random Forest

In [6]:
forest_model = build_model_pipeline(
    train_df,
    RandomForestClassifier(
        n_estimators=120,
        max_depth=14,
        n_jobs=1,
        random_state=RANDOM_STATE,
    ),
)
forest_model.fit(x_train, y_train)
forest_metrics = evaluate_classifier(forest_model, x_val, y_val)
forest_metrics

{'accuracy': 0.8160839700547966,
 'precision': 0.8213114754098361,
 'recall': 0.42230963753863443,
 'f1': 0.5578029318983114,
 'roc_auc': 0.8894009420499874}

## Gradient Boosting

In [7]:
boosting_model = build_model_pipeline(
    train_df,
    GradientBoostingClassifier(random_state=RANDOM_STATE),
)
boosting_model.fit(x_train, y_train)
boosting_metrics = evaluate_classifier(boosting_model, x_val, y_val)
boosting_metrics

{'accuracy': 0.8353785598518175,
 'precision': 0.7575867052023122,
 'recall': 0.5892104523742624,
 'f1': 0.662873399715505,
 'roc_auc': 0.8948949729608943}

Gradient Boosting даёт лучший F1-score среди рассмотренных моделей, поэтому на этапе CP1 именно он выбирается как лучшая модель.

In [8]:
results = pd.DataFrame([
    {'model': 'logistic_regression', **logistic_metrics},
    {'model': 'knn_classifier', **knn_metrics},
    {'model': 'decision_tree', **tree_metrics},
    {'model': 'random_forest', **forest_metrics},
    {'model': 'gradient_boosting', **boosting_metrics},
]).sort_values('f1', ascending=False).reset_index(drop=True)

results

,model,accuracy,precision,recall,f1,roc_auc
0,gradient_boosting,0.835379,0.757587,0.589210,0.662873,0.894895
1,decision_tree,0.815389,0.692384,0.590053,0.637136,0.870959
2,knn_classifier,0.799028,0.669627,0.529643,0.591465,0.839172
3,logistic_regression,0.796867,0.684148,0.483844,0.566820,0.852067
4,random_forest,0.816084,0.821311,0.422310,0.557803,0.889401


In [9]:
best_model = boosting_model
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
dump(best_model, MODEL_PATH)
results.to_csv(METRICS_PATH, index=False)
MODEL_PATH, METRICS_PATH

(WindowsPath('C:/Users/user/Desktop/ml-project/hseml-group-project-ruf019/models/best_model_cp1.joblib'),
 WindowsPath('C:/Users/user/Desktop/ml-project/hseml-group-project-ruf019/data/processed/metrics_cp1.csv'))